<a href="https://colab.research.google.com/github/jini28172-cyber/LLM-FROM-SCRATCH/blob/main/LLM_ch05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install tiktoken

import torch
import torch.nn as nn
import tiktoken

# 1. 멀티헤드 어텐션 (3장)
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), "d_out은 num_heads로 나누어떨어져야 합니다."
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        queries = self.W_query(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values = self.W_value(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values).transpose(1, 2).contiguous().view(b, num_tokens, self.d_out)
        return self.out_proj(context_vec)

# 2. 층 정규화, GELU, 피드포워드 (4장)
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(torch.sqrt(torch.tensor(2.0 / torch.pi)) * (x + 0.044715 * torch.pow(x, 3))))

class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)

# 3. 트랜스포머 블록 & GPTModel (4장)
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"], d_out=cfg["emb_dim"], context_length=cfg["context_length"],
            num_heads=cfg["n_heads"], dropout=cfg["drop_rate"], qkv_bias=cfg["qkv_bias"]
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x
        x = self.drop_shortcut(self.att(self.norm1(x))) + shortcut
        shortcut = x
        x = self.drop_shortcut(self.ff(self.norm2(x))) + shortcut
        return x

class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        self.trf_blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        x = self.tok_emb(in_idx) + self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        return self.out_head(x)

# 4. 텍스트 생성 함수 (4장)
def generate_text_simple(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]
        idx_next = torch.argmax(torch.softmax(logits, dim=-1), dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

print("GPU 사용 가능 여부:", torch.cuda.is_available())
print("4장 모델 뼈대 세팅 완료!")


GPU 사용 가능 여부: True
4장 모델 뼈대 세팅 완료!


In [2]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 256,   # 5장에서는 학습 속도를 위해 1024 -> 256으로 단축
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval()

# 글자 -> 토큰 ID 텐서로 변환
def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0) # 배치 차원 추가
    return encoded_tensor

# 토큰 ID 텐서 -> 글자로 변환
def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0) # 배치 차원 제거
    return tokenizer.decode(flat.tolist())

# 작동 테스트
start_context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context, tokenizer),
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M["context_length"]
)

print("출력 결과:\n", token_ids_to_text(token_ids, tokenizer))


출력 결과:
 Every effort moves you rentingetic wasnم refres RexMeCHicular stren


In [3]:
# 토크나이저를 이용해 문장을 바로 토큰 ID 텐서로 변환
inputs = torch.tensor([
    tokenizer.encode("every effort moves"),
    tokenizer.encode("I really like")
])

targets = torch.tensor([
    tokenizer.encode(" effort moves you"),
    tokenizer.encode(" really like chocolate")
])

print("inputs:\n", inputs)
print("targets:\n", targets)



inputs:
 tensor([[16833,  3626,  6100],
        [   40,  1107,   588]])
targets:
 tensor([[ 3626,  6100,   345],
        [ 1107,   588, 11311]])


In [4]:
# Edith Wharton의 단편 소설 등의 영문 샘플 텍스트로 text_data 직접 생성 및 파일 저장 코드
text_data = (
    "I had always thought Jack Gisburn rather a cheap genius--though a good fellow enough..."
) * 60  # 전체 긴 소설 원문 및 파일 저장 코드는 참조된 웹 문서에서 확인하실 수 있습니다.

with open("the-verdict.txt", "w", encoding="utf-8") as f:
    f.write(text_data)

print("총 글자 수:", len(text_data))
print("총 토큰 수:", len(tokenizer.encode(text_data)))


총 글자 수: 5220
총 토큰 수: 1140


In [5]:
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128,
                         shuffle=True, drop_last=True, num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle,
        drop_last=drop_last, num_workers=num_workers
    )
    return dataloader

# 90% 훈련 데이터 / 10% 검증 데이터 분할
train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]

torch.manual_seed(123)

train_loader = create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

print("훈련 데이터 배치 개수:", len(train_loader))
print("검증 데이터 배치 개수:", len(val_loader))



훈련 데이터 배치 개수: 2
검증 데이터 배치 개수: 0


In [6]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
    return loss

def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches


In [7]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss

def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text_simple(
            model=model, idx=encoded, max_new_tokens=50, context_size=context_size
        )
    decoded_text = token_ids_to_text(token_ids, tokenizer)
    print(decoded_text.replace("\n", " "))
    model.train()

def train_model_simple(model, train_loader, val_loader, optimizer, device, num_epochs,
                       eval_freq, eval_iter, start_context, tokenizer):
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1

    for epoch in range(num_epochs):
        model.train()
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()
            optimizer.step()
            tokens_seen += input_batch.numel()
            global_step += 1

            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter
                )
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        # 에포크가 끝날 때마다 현재 실력으로 문장 생성해보기
        generate_and_print_sample(model, tokenizer, device, start_context)

    return train_losses, val_losses, track_tokens_seen

# GPU 장치 설정 및 학습 시작!
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)

num_epochs = 10
train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context="Every effort moves you", tokenizer=tokenizer
)


Ep 1 (Step 000000): Train loss 4.372, Val loss nan
Every effort moves you barbAdapt thought Jack Gisburn rather a good fellow enough...I had always thought Jack Gisburn rather a cheap genius--though a cheap genius--though a cheap genius--though a good fellow enough...I had always thought Jack Gisburn
Every effort moves you a cheap genius--though a cheap genius--though a good fellow enough...I had always thought Jack Gisburn rather a good fellow enough...I had always thought Jack Gisburn rather a good fellow enough...I had always thought Jack Gis
Ep 3 (Step 000005): Train loss 0.094, Val loss nan
Every effort moves you a cheap genius--though a cheap genius--though a good fellow enough...I had always thought Jack Gisburn rather a good fellow enough...I had always thought Jack Gisburn rather a good fellow enough...I had always thought Jack Gis
Every effort moves you a cheap genius--though a cheap genius--though a good fellow enough...I had always thought Jack Gisburn rather a cheap genius

In [8]:
def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=None):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]

        # 1. Top-k 샘플링: 확률이 높은 상위 k개 단어만 남기고 나머지는 모두 제외(-inf)
        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(
                logits < min_val,
                torch.tensor(float('-inf')).to(logits.device),
                logits
            )

        # 2. Temperature(온도) 조절: 0보다 크면 확률 분포에 따라 무작위로 뽑고, 0이면 무조건 1등만 선택
        if temperature > 0.0:
            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)

        # 종료 토큰(eos_id)이 나오면 문장 생성 중단
        if idx_next == eos_id:
            break

        idx = torch.cat((idx, idx_next), dim=1)

    return idx


In [11]:
torch.manual_seed(123)

token_ids = generate(
    model=model,
    idx=text_to_token_ids("Every effort moves you", tokenizer).to(device),
    max_new_tokens=15,
    context_size=GPT_CONFIG_124M["context_length"],
    top_k=25,
    temperature=1.4
)

print("출력 결과:\n", token_ids_to_text(token_ids, tokenizer))


출력 결과:
 Every effort moves you a cheap genius--though a good fellow enough...I had always thought Jack


In [12]:
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    },
    "model_and_optimizer.pth"
)
print("모델 및 옵티마이저 저장 완료!")


모델 및 옵티마이저 저장 완료!


In [13]:
checkpoint = torch.load("model_and_optimizer.pth", map_location=device, weights_only=True)

model = GPTModel(GPT_CONFIG_124M)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=0.0005, weight_decay=0.1)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
model.train()

print("모델 및 옵티마이저 복구 완료!")


모델 및 옵티마이저 복구 완료!


In [15]:
from transformers import GPT2LMHeadModel
import numpy as np

# 1. 허깅페이스에서 OpenAI의 실제 GPT-2(124M) 모델 불러오기
hf_model = GPT2LMHeadModel.from_pretrained("gpt2")
hf_sd = hf_model.state_dict()

# 2. OpenAI GPT-2(124M) 규격에 맞게 설정 업데이트
NEW_CONFIG = GPT_CONFIG_124M.copy()
NEW_CONFIG.update({"context_length": 1024, "qkv_bias": True})

gpt = GPTModel(NEW_CONFIG)
gpt.eval()

def assign(left, right):
    if left.shape != right.shape:
        raise ValueError(f"모양이 다릅니다. Left: {left.shape}, Right: {right.shape}")
    return torch.nn.Parameter(right.clone().detach())

# 3. 우리가 만든 GPTModel에 OpenAI 가중치 이식하기
with torch.no_grad():
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, hf_sd["transformer.wte.weight"])
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, hf_sd["transformer.wpe.weight"])

    for b in range(NEW_CONFIG["n_layers"]):
        # Q, K, V 가중치 및 편향 분리 후 이식 (허깅페이스는 행렬이 전치(T)되어 있으므로 .T 적용)
        q_w, k_w, v_w = hf_sd[f"transformer.h.{b}.attn.c_attn.weight"].chunk(3, dim=-1)
        gpt.trf_blocks[b].att.W_query.weight = assign(gpt.trf_blocks[b].att.W_query.weight, q_w.T)
        gpt.trf_blocks[b].att.W_key.weight = assign(gpt.trf_blocks[b].att.W_key.weight, k_w.T)
        gpt.trf_blocks[b].att.W_value.weight = assign(gpt.trf_blocks[b].att.W_value.weight, v_w.T)

        q_b, k_b, v_b = hf_sd[f"transformer.h.{b}.attn.c_attn.bias"].chunk(3, dim=-1)
        gpt.trf_blocks[b].att.W_query.bias = assign(gpt.trf_blocks[b].att.W_query.bias, q_b)
        gpt.trf_blocks[b].att.W_key.bias = assign(gpt.trf_blocks[b].att.W_key.bias, k_b)
        gpt.trf_blocks[b].att.W_value.bias = assign(gpt.trf_blocks[b].att.W_value.bias, v_b)

        # 어텐션 출력 프로젝션
        gpt.trf_blocks[b].att.out_proj.weight = assign(
            gpt.trf_blocks[b].att.out_proj.weight, hf_sd[f"transformer.h.{b}.attn.c_proj.weight"].T)
        gpt.trf_blocks[b].att.out_proj.bias = assign(
            gpt.trf_blocks[b].att.out_proj.bias, hf_sd[f"transformer.h.{b}.attn.c_proj.bias"])

        # 피드포워드(MLP) 층
        gpt.trf_blocks[b].ff.layers[0].weight = assign(
            gpt.trf_blocks[b].ff.layers[0].weight, hf_sd[f"transformer.h.{b}.mlp.c_fc.weight"].T)
        gpt.trf_blocks[b].ff.layers[0].bias = assign(
            gpt.trf_blocks[b].ff.layers[0].bias, hf_sd[f"transformer.h.{b}.mlp.c_fc.bias"])
        gpt.trf_blocks[b].ff.layers[2].weight = assign(
            gpt.trf_blocks[b].ff.layers[2].weight, hf_sd[f"transformer.h.{b}.mlp.c_proj.weight"].T)
        gpt.trf_blocks[b].ff.layers[2].bias = assign(
            gpt.trf_blocks[b].ff.layers[2].bias, hf_sd[f"transformer.h.{b}.mlp.c_proj.bias"])

        # 층 정규화(LayerNorm)
        gpt.trf_blocks[b].norm1.scale = assign(
            gpt.trf_blocks[b].norm1.scale, hf_sd[f"transformer.h.{b}.ln_1.weight"])
        gpt.trf_blocks[b].norm1.shift = assign(
            gpt.trf_blocks[b].norm1.shift, hf_sd[f"transformer.h.{b}.ln_1.bias"])
        gpt.trf_blocks[b].norm2.scale = assign(
            gpt.trf_blocks[b].norm2.scale, hf_sd[f"transformer.h.{b}.ln_2.weight"])
        gpt.trf_blocks[b].norm2.shift = assign(
            gpt.trf_blocks[b].norm2.shift, hf_sd[f"transformer.h.{b}.ln_2.bias"])

    # 최종 정규화 및 출력 헤드 (가중치 공유)
    gpt.final_norm.scale = assign(gpt.final_norm.scale, hf_sd["transformer.ln_f.weight"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, hf_sd["transformer.ln_f.bias"])
    gpt.out_head.weight = assign(gpt.out_head.weight, hf_sd["transformer.wte.weight"])

gpt.to(device)
print("🎉 진짜 OpenAI GPT-2(124M) 가중치 이식 완벽 성공!")


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

🎉 진짜 OpenAI GPT-2(124M) 가중치 이식 완벽 성공!


In [16]:
torch.manual_seed(123)

token_ids = generate(
    model=gpt,
    idx=text_to_token_ids("Every effort moves you", tokenizer).to(device),
    max_new_tokens=25,
    context_size=NEW_CONFIG["context_length"],
    top_k=50,
    temperature=1.5
)

print("출력 결과:\n", token_ids_to_text(token_ids, tokenizer))


출력 결과:
 Every effort moves you as far as the hand can go until the end of your turn unless something happens

This would remove you from a battle
